# 1. Data structuring 
process curated video data 

### environment setup

In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import cv2
from tqdm import tqdm
from ultralytics import YOLO
from tqdm import tqdm
import yaml
import random, shutil
import tempfile

#plotting
import matplotlib.pyplot as plt
from matplotlib.widgets import RectangleSelector
from ipywidgets import interact, IntSlider
import seaborn as sns
from mpl_toolkits.axes_grid1 import make_axes_locatable

from io import BytesIO
from PIL import Image



In [2]:
os.chdir('/Users/inesaitsahalia/')
print(f"Current working directory: {os.getcwd()}")

Current working directory: /Users/inesaitsahalia


In [3]:
from  Desktop.labeling_data.yolo.thermal_export import process_directory
from Desktop.labeling_data.timestamps import get_creation_time, extract_timestamps_from_folder
from Desktop.labeling_data.time_alignment import extract_ffmpeg_timecode, rgb_timecode_to_ms, csq_filename_to_ms, trim_video, trim_video_pair, match_rgb_to_thermal, trim_session
from Desktop.labeling_data.collect_thermal_mp4s import collect_thermal_mp4s
from  Desktop.labeling_data.yolo.frame_extraction import extract_frames_from_folder

In [4]:

#set up path here for the videos
SRC_ROOT_DIR = Path("Google Drive/My Drive/fieldwork_data")
TARGET_ROOT_DIR = Path("Desktop/labeling_data/data_structure")


## selection and .csq conversion (only here if you haven't already done it in in the curation step 

In [5]:
# # get the time matched rgb and thermal videos for manual checking (still need to figure out which one is the leftmost or rightmost but one step at a time) 
# match_rgb_to_thermal(
#     rgb_dir=Path(SRC_ROOT_DIR/'2024-05-18'/'GOPROA'),
#     thermal_dir=Path(SRC_ROOT_DIR/'2024-05-18'/'FLIR1'),
#     max_diff_ms = 20000    
# )

# folder = '2024_05_18-session_0006'
# process_directory(TARGET_ROOT_DIR/folder/'thermal_2', TARGET_ROOT_DIR/folder/'thermal_2', color = 'magma',preview = True, max_frames = 20000)

## temporal alignment 

In [6]:

# sessions = sorted([p for p in target_root.iterdir() if p.is_dir() and p.name.startswith("2024-05-18_")])

# for session_path in sessions:
#     for thermal_sub in ["thermal_1", "thermal_2"]:
#         video_folder = session_path / thermal_sub
#         if not video_folder.exists():
#             print(f"❌ Skipping missing folder: {video_folder}")
#             continue

#         # Create output subfolder within the thermal directory
#         output_folder = video_folder 
#         print(f"\n📁 Session: {session_path.name} — {thermal_sub}")
#         trim_session(video_folder, output_folder)
        


In [ ]:
########## OLD CODE
# #loop over sessions

# VIDEO_FOLDER = Path(TARGET_ROOT_DIR/"collected_thermal_videos")         # folder containing your .mp4 videos
# OUTPUT_FOLDER = Path(TARGET_ROOT_DIR/"extracted_frames")            # root directory for output frame folders
# extract_frames_from_folder(VIDEO_FOLDER, OUTPUT_FOLDER, target_fps = 15)



## build frames folder 
this will be used for the tracking and visualization
add a folder to each thermal subfolder for the extracted frames 

In [7]:
# Root structure
root_dir = Path("Desktop/labeling_data/data_structure")
sessions = sorted([p for p in root_dir.iterdir() if p.is_dir() and p.name.startswith("2024_05_18-session_")])
sessions

[PosixPath('Desktop/labeling_data/data_structure/2024_05_18-session_0001'),
 PosixPath('Desktop/labeling_data/data_structure/2024_05_18-session_0002'),
 PosixPath('Desktop/labeling_data/data_structure/2024_05_18-session_0003'),
 PosixPath('Desktop/labeling_data/data_structure/2024_05_18-session_0004'),
 PosixPath('Desktop/labeling_data/data_structure/2024_05_18-session_0005'),
 PosixPath('Desktop/labeling_data/data_structure/2024_05_18-session_0006')]

In [8]:

for session_path in sessions:
    for thermal_sub in ["thermal_1", "thermal_2"]:
        video_folder = session_path / thermal_sub
        if not video_folder.exists():
            print(f"❌ Skipping missing folder: {video_folder}")
            continue

        # Create output subfolder within the thermal directory
        output_folder = video_folder / "extracted_frames"
        output_folder.mkdir(parents=True, exist_ok=True)

        print(f"\n📁 Session: {session_path.name} — {thermal_sub}")
        extract_frames_from_folder(video_folder, output_folder, target_fps=15)


📁 Session: 2024_05_18-session_0001 — thermal_1
📽️ Processing time_cropped-thermal_8.0_19.0.mp4


Extracting time_cropped-thermal_8.0_19.0.mp4: 100%|██████████| 7991/7991 [00:02<00:00, 3092.87it/s]


✅ Extracted 3996 frames at 15 FPS to Desktop/labeling_data/data_structure/2024_05_18-session_0001/thermal_1/extracted_frames/time_cropped-thermal_8.0_19.0


📁 Session: 2024_05_18-session_0001 — thermal_2
📽️ Processing time_cropped-thermal_10.0_20.0.mp4


Extracting time_cropped-thermal_10.0_20.0.mp4: 100%|██████████| 8091/8091 [00:02<00:00, 2925.70it/s]


✅ Extracted 4046 frames at 15 FPS to Desktop/labeling_data/data_structure/2024_05_18-session_0001/thermal_2/extracted_frames/time_cropped-thermal_10.0_20.0


📁 Session: 2024_05_18-session_0002 — thermal_1
📽️ Processing time_cropped-thermal_7.0_21.0.mp4


...

✅ Extracted 16034 frames at 15 FPS to Desktop/labeling_data/data_structure/2024_05_18-session_0005/thermal_2/extracted_frames/time_cropped-thermal_11.0_19.0


📁 Session: 2024_05_18-session_0006 — thermal_1
📽️ Processing time_cropped-thermal_12.0_17.0.mp4


Extracting time_cropped-thermal_12.0_17.0.mp4: 100%|██████████| 20000/20000 [00:06<00:00, 2977.80it/s]


✅ Extracted 10000 frames at 15 FPS to Desktop/labeling_data/data_structure/2024_05_18-session_0006/thermal_1/extracted_frames/time_cropped-thermal_12.0_17.0


📁 Session: 2024_05_18-session_0006 — thermal_2
📽️ Processing time_cropped-thermal_13.0_18.0.mp4


Extracting time_cropped-thermal_13.0_18.0.mp4: 100%|██████████| 20000/20000 [00:06<00:00, 3284.97it/s]

✅ Extracted 10000 frames at 15 FPS to Desktop/labeling_data/data_structure/2024_05_18-session_0006/thermal_2/extracted_frames/time_cropped-thermal_13.0_18.0

